# Analisi delle Spese di Rimborso

Questo notebook dimostra come creare agenti che utilizzano plugin per elaborare le spese di viaggio da immagini di ricevute locali, generare un'email di richiesta di rimborso spese e visualizzare i dati delle spese tramite un grafico a torta. Gli agenti scelgono dinamicamente le funzioni in base al contesto del compito.

Passi:
1. L'agente OCR elabora l'immagine locale della ricevuta ed estrae i dati delle spese di viaggio.
2. L'agente Email genera un'email di richiesta di rimborso spese.

### Esempio di uno scenario di spesa di viaggio:
Immagina di essere un dipendente che viaggia per una riunione di lavoro in un'altra città. La tua azienda ha una politica di rimborso per tutte le spese ragionevoli relative al viaggio. Ecco una suddivisione delle potenziali spese di viaggio:
- Trasporto:
Biglietto aereo per un viaggio di andata e ritorno dalla tua città di origine alla città di destinazione.
Taxi o servizi di ride-hailing da e per l'aeroporto.
Trasporto locale nella città di destinazione (come trasporto pubblico, auto a noleggio o taxi).

- Alloggio:
Soggiorno in hotel per tre notti in un hotel business di fascia media vicino alla sede della riunione.

- Pasti:
Indennità giornaliera per colazione, pranzo e cena, basata sulla politica aziendale del per diem.

- Spese Varie:
Tariffe di parcheggio all'aeroporto.
Costi di accesso a internet in hotel.
Mance o piccoli costi di servizio.

- Documentazione:
Devi presentare tutte le ricevute (voli, taxi, hotel, pasti, ecc.) e un rapporto spese completo per il rimborso.


## Importa le librerie necessarie

Importa le librerie e i moduli necessari per il notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Definire i Modelli di Spesa

 Crea un modello Pydantic per le singole spese e una classe ExpenseFormatter per convertire una query utente in dati di spesa strutturati.

 Ogni spesa sarà rappresentata nel formato:
 `{'date': '07-Mar-2025', 'description': 'volo verso la destinazione', 'amount': 675.99, 'category': 'Trasporto'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Generazione dell'Email

Crea una funzione strumento per generare un'email per l'invio di una richiesta di rimborso spese.
- Questo strumento utilizza il decoratore `@tool` del Microsoft Agent Framework.
- Calcola l'importo totale delle spese e formatta i dettagli nel corpo dell'email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Strumento per l'Estrazione delle Spese di Viaggio dalle Immagini delle Ricevute

Crea una funzione strumento per estrarre le spese di viaggio dalle immagini delle ricevute.
- Questo strumento utilizza il decoratore `@tool` del Microsoft Agent Framework.
- Legge l'immagine della ricevuta, la codifica in base64 e restituisce l'URI dei dati per l'agente da analizzare.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Elaborazione delle Spese

Definire gli agenti e collegarli in un flusso di lavoro sequenziale utilizzando `WorkflowBuilder`.
- L'agente OCR estrae dati di spesa strutturati dall'immagine della ricevuta utilizzando lo strumento `load_receipt_image`.
- L'agente Email prende i dati estratti e genera una email professionale per la richiesta di rimborso spese utilizzando lo strumento `generate_expense_email`.
- `WorkflowBuilder` con `add_edge` crea una pipeline sequenziale: Agente OCR → Agente Email.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Funzione principale

Costruisci il flusso di lavoro sequenziale ed eseguilo per elaborare l'immagine della ricevuta e generare l'email per la richiesta di rimborso spese.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Pur impegnandoci per garantire l’accuratezza, si prega di notare che le traduzioni automatizzate possono contenere errori o inesattezze. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, è consigliata la traduzione professionale effettuata da un umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
